In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

SEED = 42
np.random.seed(SEED)

# Загрузка данных 
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val = pd.read_csv('../data/processed/X_val.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# 2. Типы колонок
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Числовых признаков: {len(numeric_cols)}")
print(f"Категориальных: {len(categorical_cols)}")
if categorical_cols:
    print("Примеры категориальных:", categorical_cols[:5])

# Препроцессор
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

def get_pipeline(model):
    return Pipeline([('prep', preprocessor), ('reg', model)])

# Модели
models = {
    'Ridge': Ridge(alpha=1.0, random_state=SEED),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=SEED, max_iter=5000),
    'RandomForest': RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=SEED
    )
}

# Обучение и сбор метрик
results = []
best_model = None
best_r2 = -np.inf

print("\nОбучение моделей:")
print("-" * 60)
for name, model in models.items():
    pipe = get_pipeline(model)
    pipe.fit(X_train, y_train)
    
    # Предсказания на валидации
    y_pred_val = pipe.predict(X_val)
    r2 = r2_score(y_val, y_pred_val)
    mae = mean_absolute_error(y_val, y_pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    
    results.append({
        'Model': name,
        'R² (val)': r2,
        'MAE (val)': mae,
        'RMSE (val)': rmse
    })
    
    print(f"{name:15} | R² = {r2:.4f} | MAE = {mae:.2f} | RMSE = {rmse:.2f}")
    
    if r2 > best_r2:
        best_r2 = r2
        best_model = pipe
    
    joblib.dump(pipe, f'../models/{name.lower()}_model.pkl')

# Оценка лучшей модели на тесте 
y_pred_test = best_model.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("\n" + "=" * 60)
print(f"ЛУЧШАЯ МОДЕЛЬ (по R² на валидации):")
print(f"  На ТЕСТЕ: R² = {r2_test:.4f}, MAE = {mae_test:.2f}, RMSE = {rmse_test:.2f}")
print("=" * 60)

# Сохраняем лучшую модель
joblib.dump(best_model, '../models/best_model.pkl')

# Сводная таблица результатов
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R² (val)', ascending=False)
print("\nСводная таблица результатов на валидации:")
print(results_df.to_string(index=False))

# Дополнительно сохраняем таблицу в CSV
results_df.to_csv('../models/model_results.csv', index=False)
print("\nТаблица результатов сохранена в ../models/model_results.csv")

/var/folders/t9/w50p8w0917b0t61ks_6y_3g40000gn/T/ipykernel_24907/99545330.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Числовых признаков: 9
Категориальных: 7
Примеры категориальных: ['order_channel', 'store_location_type', 'region', 'customer_age_group', 'customer_gender']

Обучение моделей:
------------------------------------------------------------
Ridge           | R² = 0.9501 | MAE = 1.00 | RMSE = 1.23
ElasticNet      | R² = 0.9500 | MAE = 1.00 | RMSE = 1.23
RandomForest    | R² = 0.9526 | MAE = 0.97 | RMSE = 1.19
GradientBoosting | R² = 0.9533 | MAE = 0.97 | RMSE = 1.19

ЛУЧШАЯ МОДЕЛЬ (по R² на валидации):
  На ТЕСТЕ: R² = 0.9549, MAE = 0.97, RMSE = 1.18

Сводная таблица результатов на валидации:
           Model  R² (val)  MAE (val)  RMSE (val)
GradientBoosting  0.953276   0.966718    1.185269
    RandomForest  0.952553   0.971392    1.194398
           Ridge  0.950061   0.996143    1.225373
      ElasticNet  0.950024   0.996074    1.225818

Таблица результатов сохранена в ../models/model_results.csv
